In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [2]:
import math

def u_exact(x):
    # x: numpy array
    return np.sin(np.pi * x)

def f_rhs(x):
    # f(x) = -pi^2 sin(pi x)
    return - (math.pi ** 2) * np.sin(math.pi * x)


In [3]:
class PINN(nn.Module):
    def __init__(self, layers):
        super(PINN, self).__init__()
        # layers is a list like [1, 32, 32, 1]
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
        
        # Initialize weights (Xavier initialization is common)
        for m in self.layers:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x shape: (N, 1)
        z = x
        for i, layer in enumerate(self.layers[:-1]):
            z = torch.tanh(layer(z))
        # Last layer without activation
        z = self.layers[-1](z)
        return z


In [4]:
layers = [1, 32, 32, 32, 1]  # input 1D, three hidden layers, output 1D
model = PINN(layers).to(device)
model


PINN(
  (layers): ModuleList(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=True)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [5]:
# Number of collocation points
N_f = 1000

# Interior points in (0,1)
x_f = np.random.rand(N_f, 1)  # uniform in [0,1]
x_f = x_f.astype(np.float32)

# Boundary points: x=0 and x=1
x_b = np.array([[0.0], [1.0]], dtype=np.float32)
u_b = np.array([[0.0], [0.0]], dtype=np.float32)  # boundary values

# Convert to torch tensors
x_f_t = torch.tensor(x_f, requires_grad=True).to(device)
x_b_t = torch.tensor(x_b, requires_grad=True).to(device)
u_b_t = torch.tensor(u_b).to(device)


In [6]:
def pde_residual(x):
    """
    Compute PDE residual r_theta(x) = u''(x) + pi^2 sin(pi x)
    x: tensor of shape (N, 1) with requires_grad=True
    """
    # Forward pass: u(x)
    u = model(x)

    # First derivative du/dx
    du_dx = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]

    # Second derivative d2u/dx2
    d2u_dx2 = torch.autograd.grad(
        du_dx, x,
        grad_outputs=torch.ones_like(du_dx),
        create_graph=True,
        retain_graph=True
    )[0]

    # Right-hand side term: -pi^2 sin(pi x)
    f = - (math.pi ** 2) * torch.sin(math.pi * x)

    # Residual: u''(x) - f(x) = u''(x) + pi^2 sin(pi x)
    r = d2u_dx2 - f
    return r


In [7]:
def loss_function(x_f, x_b, u_b, lambda_bc=1.0):
    # PDE residual loss
    r = pde_residual(x_f)
    loss_pde = torch.mean(r**2)

    # Boundary loss
    u_pred_b = model(x_b)
    loss_bc = torch.mean((u_pred_b - u_b)**2)

    loss = loss_pde + lambda_bc * loss_bc
    return loss, loss_pde, loss_bc


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5000
lambda_bc = 1.0

history_total = []
history_pde = []
history_bc = []

for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    loss, loss_pde, loss_bc = loss_function(x_f_t, x_b_t, u_b_t, lambda_bc=lambda_bc)
    
    loss.backward()
    optimizer.step()
    
    history_total.append(loss.item())
    history_pde.append(loss_pde.item())
    history_bc.append(loss_bc.item())
    
    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Loss: {loss.item():.4e} | "
              f"PDE: {loss_pde.item():.4e} | "
              f"BC: {loss_bc.item():.4e}")


Epoch 500/5000 | Loss: 3.6466e-03 | PDE: 3.5290e-03 | BC: 1.1759e-04
Epoch 1000/5000 | Loss: 4.4277e-04 | PDE: 4.3598e-04 | BC: 6.7908e-06
Epoch 1500/5000 | Loss: 1.5096e-04 | PDE: 1.5048e-04 | BC: 4.8397e-07
Epoch 2000/5000 | Loss: 1.0904e-04 | PDE: 1.0888e-04 | BC: 1.5636e-07
Epoch 2500/5000 | Loss: 9.1228e-05 | PDE: 9.1161e-05 | BC: 6.6861e-08
Epoch 3000/5000 | Loss: 5.0289e-04 | PDE: 4.7870e-04 | BC: 2.4186e-05
Epoch 3500/5000 | Loss: 7.1965e-05 | PDE: 7.1949e-05 | BC: 1.5727e-08
Epoch 4000/5000 | Loss: 6.4873e-05 | PDE: 6.4849e-05 | BC: 2.3338e-08


In [ ]:
plt.figure(figsize=(8,5))
plt.semilogy(history_total, label='Total loss')
plt.semilogy(history_pde, label='PDE loss')
plt.semilogy(history_bc, label='BC loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.legend()
plt.title('Training loss history')
plt.grid(True)
plt.show()


In [ ]:
# Evaluation grid
x_eval = np.linspace(0, 1, 200).reshape(-1, 1).astype(np.float32)
x_eval_t = torch.tensor(x_eval, requires_grad=True).to(device)

# PINN prediction
with torch.no_grad():
    u_pred = model(x_eval_t).cpu().numpy()

# Exact solution
u_true = u_exact(x_eval)

plt.figure(figsize=(8,5))
plt.plot(x_eval, u_true, 'k-', label='Exact solution')
plt.plot(x_eval, u_pred, 'r--', label='PINN prediction')
plt.xlabel('x')
plt.ylabel('u(x)')
plt.legend()
plt.title('PINN vs exact solution')
plt.grid(True)
plt.show()